# Lab 13 - Web IQ authentication with Microsoft Entra ID

## Required permissions

- **You:** an approved caller application's client/tenant IDs and authorized short-lived credential for the local client-secret exercise, or access to Azure compute exposing the selected managed identity. This is app-only authentication; `az login` or Azure Reader alone does not authorize requests.
- **Facilitator/app owner and Web IQ profile manager:** prepare the calling app/identity and approved credential, and bind its **application/client ID** to the Web IQ profile. Participants only use that prepared identity; the notebook does not create apps or grant permissions. Respect tenant credential policies.

**Attendee alias:** keep your `WORKSHOP_PREFIX` in your own `.env`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). This notebook only acquires tokens and makes requests; it creates no named resources to prefix. Do not prepend aliases to existing client IDs, tenant IDs, scopes or profile bindings. Each attendee uses their own kernel and approved credentials. Aliases prevent naming collisions, not enforce access isolation.

**Advanced, optional lab.** Keep lab 01 as the API-key introduction. Here, replace the shared key with an app-only bearer token, bind the caller's client ID to a Web IQ profile, and understand which identity a Foundry-hosted agent should use. Allow 25-35 minutes.

Architecture: **application identity -> Entra token for Web IQ -> Web IQ AppID binding -> Web IQ API**. No Foundry deployment or model call is needed for the local exercise. Search requests incur Web IQ usage charges; use public, non-sensitive queries.

Prerequisites: a working Web IQ profile, permission to manage its client-ID bindings, and an Entra app registration with a short-lived client secret for the local exercise. Use your existing workshop environment and packages. If your organization disallows client secrets, use an approved workload identity on its hosting platform instead; do not weaken policy.

References: [Web IQ authentication](https://learn.microsoft.com/web-iq/get-started/authentication), [hosted-agent identity FAQ](https://learn.microsoft.com/web-iq/resources/faq#api-authentication).

## 1. Bind the caller, not the Azure resource

| Identifier | Meaning |
|---|---|
| Entra application (client) ID | The calling app or workload identity. Bind this value in Web IQ. |
| Object/principal ID | The directory object used for many RBAC assignments. Do not enter this as the client ID. |
| Web IQ AppID | The Web IQ profile that owns the service access and quota. This is not the Entra client ID. |
| Foundry resource/project ID | An Azure resource identifier, not the caller's client ID. |

For the local exercise, create or select a dedicated **Entra app registration**, record its Application (client) ID and Directory (tenant) ID, and create a short-lived client secret. Copy the secret **value**, not its secret ID. The next section prompts for it without echoing or saving it.

In [Web IQ Profile Management](https://webiq.microsoft.ai/profiles), select the intended profile, open **Application (Client) IDs**, choose **Bind Application (Client) ID**, and enter the caller's client ID. Allow up to one minute for synchronization. If this tab is unavailable for your trial, request Entra authentication access from your Microsoft representative.

Binding authorizes that identity for the profile; acquiring a token alone does not. This is Web IQ authorization, not an Azure RBAC role on the billing resource. A developer's interactive `az login` identity is not a substitute for this app-only flow.

## 2. Choose the identity that will actually make the request

Set `WEBIQ_CLIENT_ID` in `.env` to the client ID you bound. Set `WEBIQ_TENANT_ID` for the local app registration, or use the deployment's `AZURE_TENANT_ID`. Existing environment variables take precedence.

`WEBIQ_AUTH_MODE` defaults to `client_secret` for a local notebook. Use `system_assigned` or `user_assigned` **only on Azure compute that exposes that managed identity**; registering its client ID does not let a local notebook impersonate it. For all modes, `WEBIQ_CLIENT_ID` identifies the bound caller; system-assigned identity selection itself does not take a client ID.

If the same identity is bound to multiple profiles, set `WEBIQ_APP_ID` to the intended **Web IQ AppID**. It is optional for a single binding, required for multiple bindings, and must never be added to an API-key request.

In [ ]:
import getpass
import json
import os
from pathlib import Path

import requests
from azure.identity import ClientSecretCredential, ManagedIdentityCredential
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open this notebook from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')

auth_mode = os.getenv('WEBIQ_AUTH_MODE', 'client_secret').strip()
client_id = os.getenv('WEBIQ_CLIENT_ID', '').strip()
tenant_id = (os.getenv('WEBIQ_TENANT_ID') or os.getenv('AZURE_TENANT_ID', '')).strip()
webiq_app_id = os.getenv('WEBIQ_APP_ID', '').strip()
scope = 'https://api.microsoft.ai/.default'


In [ ]:
if auth_mode not in ('client_secret', 'system_assigned', 'user_assigned'):
    raise ValueError('WEBIQ_AUTH_MODE must be client_secret, system_assigned or user_assigned.')
if not client_id:
    raise ValueError('Set WEBIQ_CLIENT_ID to the caller client ID bound in Web IQ.')

if auth_mode == 'client_secret':
    if not tenant_id:
        raise ValueError('Set WEBIQ_TENANT_ID or AZURE_TENANT_ID for the app registration.')
    client_secret = getpass.getpass('Entra app client secret value (not saved): ')
    try:
        if not client_secret.strip():
            raise ValueError('A client secret value is required for the local app-only exercise.')
        credential = ClientSecretCredential(
            tenant_id=tenant_id, client_id=client_id, client_secret=client_secret,
        )
    finally:
        del client_secret
elif auth_mode == 'user_assigned':
    credential = ManagedIdentityCredential(client_id=client_id)
else:
    credential = ManagedIdentityCredential()

print('Authentication mode:', auth_mode)
print('Expected bound caller client ID:', client_id)
print('Token scope:', scope)
print('Web IQ profile:', webiq_app_id or '(resolved from the single client-ID binding)')

## 3. Acquire an app-only token and call Web Search

The request body and endpoint are the same as lab 01. Authentication changes from `x-apikey` to `Authorization: Bearer ...`; no Web IQ API key is read or sent. The token must target **Web IQ**, not Microsoft Graph, Azure management or Foundry.

Call `get_token` before each request; Azure Identity caches and refreshes tokens as needed. Do not print the token, credential, request headers or HTTP debug logs. The cell displays only the search response body. Clear outputs before sharing notebooks, and do not share exception dumps containing credentials.

In [ ]:
question = 'recent advances in battery recycling'
access_token = credential.get_token(scope)
try:
    response = requests.post(
        'https://api.microsoft.ai/v3/search/web',
        headers={'Authorization': f'Bearer {access_token.token}', 'Content-Type': 'application/json'},
        params={'APPID': webiq_app_id} if webiq_app_id else {},
        json={
            'query': question, 'maxResults': 3, 'language': 'en', 'region': 'US',
            'contentFormat': 'passage', 'maxLength': 3000,
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    web = response.json()
    del response
finally:
    del access_token
print(json.dumps(web, indent=2, ensure_ascii=False))

## 4. Apply the same binding to a Foundry-hosted agent

Yes, Foundry is an example of a caller, but **bind the identity actually used by the outbound Web IQ request**. Web IQ supports managed identities (system- or user-assigned) and Agent Identity client IDs. The hosting platform determines how that identity obtains its app-only token.

For an Azure workload using managed identity, bind its client ID, run the managed-identity mode on that workload, request the same Web IQ scope, and use the same bearer-header request. No client secret is needed. A project identity, agent identity and hosting-compute identity are not interchangeable; verify which one the outbound credential uses before binding it. Agent Identity requires its platform-supported credential flow, not simply passing its ID to `ManagedIdentityCredential`.

**Binding does not reconfigure a Foundry connection.** Lab 02's `CustomKeys` MCP connection still sends `x-apikey`, and lab 11's agentic connection targets Azure Search, not Web IQ. This lab does not modify either connection or deploy an agent. When moving a remote MCP tool to Entra authentication, configure the supported connection/credential flow for Web IQ's audience and bind the resulting caller identity; do not paste a short-lived token into a permanent connection.

For a hosted agent that calls the REST API directly, place the token acquisition and HTTP request in its tool code, not in the model prompt. Never expose the bearer token as a tool argument or result.

## 5. Verify the boundary and troubleshoot

| Result | Check |
|---|---|
| Token acquisition fails | Tenant, client ID, secret value/expiry, or availability of the selected managed identity on the host. This happens before Web IQ authorization. |
| HTTP 401 | Token audience/expiry and client-ID binding. An identity with no AppID binding returns `AuthUnauthorizedEntryId`. Allow time for binding synchronization. |
| HTTP 400 `AuthAppIdSelectionRequired` | The identity has several bindings. Supply the intended Web IQ AppID in `WEBIQ_APP_ID`. |
| HTTP 403 `AuthUnauthorizedAppId` | The selected AppID is not bound to this caller. Other 403s can indicate profile service restrictions. |
| HTTP 429 | Profile quota/rate limit; changing authentication does not bypass it. |

**Acceptance criteria:** a successful Web Search response using only a bearer token; an explanation of the difference between client ID, principal ID and Web IQ AppID; and identification of the actual outbound identity for a Foundry-hosted deployment. Empty search results alone are not an authentication failure.

Optional facilitator check: use a separate, deliberately unbound test app and confirm that a valid token cannot access the profile. Do not remove a shared workshop or production binding to demonstrate denial. This notebook does not automatically retry, fall back to an API key, create apps or grant permissions.

Cleanup: close the credential below, restart the kernel to release in-memory credentials, delete the short-lived test secret, and remove only the dedicated test binding/app if no longer needed. Do not remove identities or bindings used by other labs or workloads.

In [ ]:
credential.close()
del credential
print('Credential closed. Restart the kernel and remove dedicated test credentials when finished.')